# 03 — RAG Career Mentor Prototype

This notebook exercises the same `ask()` function used by the production mentor pipeline.

For each question, it shows:
1. the retrieved job/career-note evidence,
2. the final mentor answer,
3. the pipeline trace returned by the mentor.

This is a developer exploration notebook, not a replacement for the full evaluation script.

In [1]:
from pathlib import Path
import sys

# Find the SmartHire project root so these notebooks work whether Jupyter
# is launched from the project root or from the notebooks/ directory.
HERE = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "src" / "config.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the SmartHire project root. "
        "Open this notebook from inside the project repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\harsh\Downloads\Final Gen ai project output\smarthire-genai-final


## Test questions

The questions below cover different RAG behaviors: skill prioritisation, career progression, and an out-of-scope request that should be blocked by the guardrails.

In [2]:
QUESTIONS = [
    "What skills should I prioritise to become a Data Scientist?",
    "What is a practical career path for becoming a Software Engineer?",
    "Can you give me a medical diagnosis for persistent chest pain?",
]

for i, question in enumerate(QUESTIONS, start=1):
    print(f"{i}. {question}")

1. What skills should I prioritise to become a Data Scientist?
2. What is a practical career path for becoming a Software Engineer?
3. Can you give me a medical diagnosis for persistent chest pain?


In [3]:
from src.mentor.rag_chain import _retrieve, ask

def run_mentor_case(question: str) -> None:
    print("=" * 100)
    print("QUESTION")
    print(question)
    print()

    # Show the evidence that the production mentor is expected to ground
    # its answer in. Out-of-scope questions are guarded before retrieval.
    evidence = _retrieve(
        question,
        profile=None,
        context=None,
        k=5,
    )

    print("RETRIEVED EVIDENCE")
    if evidence:
        for i, item in enumerate(evidence, start=1):
            print(
                f"[S{i}] {item['title']} | "
                f"{item['type']} | "
                f"relevance={item['relevance']}"
            )
            print(item["text"][:1800].strip())
            print()
    else:
        print("(No retrieval evidence.)")
        print()

    response = ask(
        question,
        None,
        [],
        None,
    )

    print("FINAL ANSWER")
    print(response["answer"])
    print()

    print("SOURCES")
    for source in response.get("sources", []):
        print(source)

    print()
    print("PIPELINE TRACE")
    print(" → ".join(response.get("pipeline_trace", [])))
    print()


for question in QUESTIONS:
    run_mentor_case(question)

QUESTION
What skills should I prioritise to become a Data Scientist?

RETRIEVED EVIDENCE
[S1] Data Science | Career note | relevance=100
# Data Science Career Note

This local note is used by the SmartHire mentor as retrieval evidence. It is project knowledge, not a live labor-market report.

## Core skill themes in the included corpus

The included Data Scientist corpus contains postings referring to Python, statistics, machine learning, NLP, deep learning, analytics, cloud platforms and data-processing tools. Exact requirements differ by posting. The mentor must not infer salary, hiring probability or a specific requirement unless the retrieved evidence supports it.

## Conceptual Data Scientist career path

This is a conceptual progression for mentoring, not a claim that every employer uses the same title ladder.

### Stage 1 — Foundation
Build proficiency in Python, SQL, statistics, data cleaning, exploratory data analysis and data visualization. The goal is to become comfortable t

## Notes on the results

For the in-scope questions, the final answer should be supported by the retrieved evidence and should use the `[S1]`, `[S2]`, etc. source labels.

For the deliberately out-of-scope medical question, the guardrail should stop the request before retrieval and Gemini generation. The expected trace is therefore **`Guardrail → Rejected`**.

This notebook is intentionally exploratory. Use `src/evaluate.py` for the formal retrieval, answer-quality, prompt-comparison, and refusal evaluation.